# Block-wise FP8 (E4M3) weight quantization

FP8 E4M3 has 4 exponent bits and 3 mantissa bits: it covers $\pm 448$ with about two significant digits, finer near zero and coarser near the top. Recent models store weights in FP8 with one scale per square **block** (for example $128 \times 128$), so each block is stretched to use the format's range.

Write `fp8_blockwise(W, block)` in PyTorch.

## Inputs
- `W`: float32 `(M, N)`. `block`: int $b \ge 1$. $M$ and $N$ need **not** be multiples of $b$: edge blocks are smaller.
- Block $(i, j)$ is `W[i*b:(i+1)*b, j*b:(j+1)*b]`; there are $\lceil M/b \rceil \times \lceil N/b \rceil$ of them.

## Per block (all arithmetic in float32)
1. $s = \max |w| / 448$; use $s = 1$ for an all-zero block.
2. Divide the block by $s$ and cast to `torch.float8_e4m3fn` (PyTorch rounds to the nearest representable value, ties to even). Dividing first guarantees every value lies in $[-448, 448]$.
3. Dequantize: cast back to float32 and multiply by $s$.

## Output
A tuple `(W_hat, scales)`: `W_hat` float32 `(M, N)` and `scales` float32 `(ceil(M/b), ceil(N/b))`. NumPy has no FP8 type, so return the dequantized weights rather than the FP8 tensor. Use tensor ops (padding is fine), no loops over blocks.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests. It needs PyTorch 2.1 or newer for the FP8 dtype.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def fp8_blockwise(W, block):
    """W float32 (M, N). Return (W_hat float32 (M, N), scales float32 (ceil(M/b), ceil(N/b)))."""
    s = W.abs().max() / 448                       # one scale for everything: wrong granularity
    return (W / s).to(torch.float8_e4m3fn).to(torch.float32) * s, s.reshape(1, 1)


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[448.0, 1.0], [-0.30000001192092896, 224.0]], "dtype": "float32"}, 2], "e": {"__tup__": [{"__nd__": [[448.0, 1.0], [-0.3125, 224.0]], "dtype": "float32"}, {"__nd__": [[1.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 2.0, 3.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.5]], "dtype": "float32"}, 2], "e": {"__tup__": [{"__nd__": [[1.0, 2.0, 3.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.5]], "dtype": "float32"}, {"__nd__": [[0.004464285913854837, 0.0066964286379516125], [1.0, 0.0011160714784637094]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[60.0, 0.5181635022163391, -0.11009939759969711, -0.08903376013040543, 0.5025036334991455, 0.5105738639831543, 0.34507566690444946, -1.4208372831344604], [-0.5697727203369141, -0.3065755069255829, 0.7395539879798889, 0.04545522481203079, -0.3938700556755066, 0.3775947391986847, 0.32589447498321533, -1.1415613889694214], [0.5414602160453796, -0.42470860481262207, 0.21265631914138794, 0.46147429943084717, -0.9584651589393616, 1.2063082456588745, -1.009986400604248, 0.8593730926513672], [0.6420454382896423, -0.17440564930438995, 1.2201077938079834, -0.617811918258667, -1.3640137910842896, 0.02170197106897831, -0.9355947375297546, -0.07422322034835815], [0.2685922384262085, 1.52106511592865, -0.2289394736289978, 0.10297729074954987, 0.6583266258239746, 0.7939301133155823, -0.11635640263557434, 0.9060273170471191], [-1.0007978677749634, -0.5254555940628052, -0.6878044605255127, -0.3192591667175293, 0.5993638038635254, 0.01907079480588436, 0.9518969655036926, -1.3533518314361572]], "dtype": "float32"}, 4], "e": {"__tup__": [{"__nd__": [[60.0, 0.5022321343421936, -0.108816958963871, -0.0920758917927742, 0.5074418783187866, 0.5074418783187866, 0.3552093207836151, -1.4208372831344604], [-0.6026785373687744, -0.3013392686843872, 0.7366071343421936, 0.0460379458963871, -0.4059534966945648, 0.38058140873908997, 0.32983720302581787, -1.1163721084594727], [0.5357142686843872, -0.435267835855484, 0.217633917927742, 0.46875, -0.913395345211029, 1.217860460281372, -1.0148837566375732, 0.8119069933891296], [0.6696428060531616, -0.1674107015132904, 1.2053570747375488, -0.6026785373687744, -1.3193488121032715, 0.022200582548975945, -0.913395345211029, -0.07611627876758575], [0.2716187834739685, 1.52106511592865, -0.21729502081871033, 0.10185704380273819, 0.6766759157180786, 0.7733439207077026, -0.12083499133586884, 0.8700119256973267], [-0.9778276085853577, -0.543237566947937, -0.7062088251113892, -0.3259425163269043, 0.5800079107284546, 0.01963568478822708, 0.9666799306869507, -1.3533518314361572]], "dtype": "float32"}, {"__nd__": [[0.1339285671710968, 0.0031715116929262877], [0.003395234700292349, 0.0030208746902644634]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[60.0, 0.5181635022163391, -0.11009939759969711, -0.08903376013040543, 0.5025036334991455, 0.5105738639831543, 0.34507566690444946, -1.4208372831344604], [-0.5697727203369141, -0.3065755069255829, 0.7395539879798889, 0.04545522481203079, -0.3938700556755066, 0.3775947391986847, 0.32589447498321533, -1.1415613889694214], [0.5414602160453796, -0.42470860481262207, 0.21265631914138794, 0.46147429943084717, -0.9584651589393616, 1.2063082456588745, -1.009986400604248, 0.8593730926513672], [0.6420454382896423, -0.17440564930438995, 1.2201077938079834, -0.617811918258667, -1.3640137910842896, 0.02170197106897831, -0.9355947375297546, -0.07422322034835815], [0.2685922384262085, 1.52106511592865, -0.2289394736289978, 0.10297729074954987, 0.6583266258239746, 0.7939301133155823, -0.11635640263557434, 0.9060273170471191], [-1.0007978677749634, -0.5254555940628052, -0.6878044605255127, -0.3192591667175293, 0.5993638038635254, 0.01907079480588436, 0.9518969655036926, -1.3533518314361572]], "dtype": "float32"}, 8], "e": {"__tup__": [{"__nd__": [[60.0, 0.5022321343421936, -0.108816958963871, -0.0920758917927742, 0.5022321343421936, 0.5022321343421936, 0.3348214030265808, -1.4732142686843872], [-0.6026785373687744, -0.3013392686843872, 0.7366071343421936, 0.0460379458963871, -0.4017857015132904, 0.3683035671710968, 0.3348214030265808, -1.2053570747375488], [0.5357142686843872, -0.435267835855484, 0.217633917927742, 0.46875, -0.9375, 1.2053570747375488, -1.0044642686843872, 0.870535671710968], [0.6696428060531616, -0.1674107015132904, 1.2053570747375488, -0.6026785373687744, -1.3392856121063232, 0.0209263376891613, -0.9375, -0.0753348171710968], [0.2678571343421936, 1.4732142686843872, -0.234375, 0.1004464253783226, 0.6696428060531616, 0.8035714030265808, -0.1171875, 0.9375], [-1.0044642686843872, -0.5357142686843872, -0.6696428060531616, -0.3348214030265808, 0.6026785373687744, 0.0188337042927742, 0.9375, -1.3392856121063232]], "dtype": "float32"}, {"__nd__": [[0.1339285671710968]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.005711071193218231, 0.00748617434874177, -0.0013961123768240213, -0.003699210472404957, -0.019281774759292603, 0.006268024444580078, -0.010874695144593716], [0.00413847342133522, -0.010303705930709839, -0.005407229997217655, 0.014969026669859886, -0.010131771676242352, -0.009171861223876476, -0.0021792398765683174], [-0.01815965585410595, -0.00012320639507379383, 0.0002520826819818467, -0.0029935531783849, -0.008431467227637768, -0.0010252752108499408, -0.006760656367987394], [-0.0035648676566779613, -0.008273131214082241, 0.005594516638666391, -0.004161816090345383, 0.00919689703732729, -0.008695139549672604, -0.0030105828773230314], [-0.007113582454621792, 0.0042039151303470135, 0.010969128459692001, 0.007305813953280449, 0.015651866793632507, 0.005025708116590977, -0.015536409802734852]], "dtype": "float32"}, 3], "e": {"__tup__": [{"__nd__": [[-0.005837032105773687, 0.007782709784805775, -0.0014592580264434218, -0.0037874914705753326, -0.019281774759292603, 0.006197713315486908, -0.010874695144593716], [0.004215634427964687, -0.010376946069300175, -0.005188473034650087, 0.01514996588230133, -0.01032952219247818, -0.008952252566814423, -0.002136100782081485], [-0.01815965585410595, -0.00012160484038759023, 0.00024320968077518046, -0.003098856657743454, -0.008263617753982544, -0.001032952219247818, -0.006990875583142042], [-0.003525791224092245, -0.00861860066652298, 0.005484564229846001, -0.004192464053630829, 0.008943923749029636, -0.008943923749029636, -0.003051794832572341], [-0.00705158244818449, 0.00430930033326149, 0.010969128459692001, 0.007266938220709562, 0.015651866793632507, 0.0050309570506215096, -0.015536410734057426]], "dtype": "float32"}, {"__nd__": [[4.053494558320381e-05, 4.3039675801992416e-05, 2.42738733504666e-05], [2.4484661480528302e-05, 3.493720214464702e-05, 3.467948772595264e-05]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}, 2], "e": {"__tup__": [{"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [[1.0, 1.0], [1.0, 1.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.037975549697876, 0.9836621880531311, 0.15725287795066833, -3.3288421630859375, 0.4927999675273895, -2.3758108615875244, 1.4371793270111084, -0.14003504812717438, 0.6289031505584717, -4.865070819854736, 0.697211742401123, -5.6296892166137695, 0.3965730667114258, 2.820499897003174, 0.6155651211738586, 0.021284805610775948], [1.5176138877868652, -1.3487528562545776, 1.0661308765411377, 1.1518895626068115, -0.6891080141067505, 0.758454442024231, -0.5475846529006958, 1.0074080228805542, -6.051620960235596, -1.1312599182128906, -1.0008971691131592, -0.04142933711409569, -0.35113951563835144, -2.245445489883423, -1.7298827171325684, 0.7347142100334167], [0.6619362235069275, -3.433306932449341, -0.11361481994390488, 0.35454902052879333, -1.5889233350753784, 0.19702233374118805, 0.5224698781967163, -0.06567299365997314, -0.6359816789627075, -1.685166597366333, -1.5041723251342773, -0.9135793447494507, 4.751058578491211, 0.2188880741596222, 2.428105115890503, -1.956758737564087], [-0.8431999683380127, -0.08240382373332977, -1.0751972198486328, -0.24874156713485718, 0.1313094198703766, -0.5801637172698975, -0.8648830056190491, 0.9216779470443726, -0.6271197199821472, 1.347702145576477, 0.13761550188064575, 0.35214996337890625, -0.2442895621061325, 0.1541779488325119, 5.416962623596191, -8.125448226928711], [-0.64524906873703, -1.1362944841384888, 6.310157299041748, 0.37474384903907776, -2.2176971435546875, 1.6373220682144165, -14.08481502532959, -0.16114665567874908, -0.6503689885139465, 0.6688022613525391, 0.45058274269104004, -1.9843556880950928, 0.25678205490112305, 0.20903705060482025, -0.5203014612197876, 0.2785908877849579], [0.2320917695760727, 2.9037036895751953, 0.1808261126279831, -0.9563969373703003, 0.8315689563751221, -1.2177550792694092, 1.463533639907837, -0.3391572833061218, 0.09288480877876282, -6.9676923751831055, 0.7631260752677917, 1.82522714138031, -0.853420078754425, -0.3174954354763031, 0.9743141531944275, -0.4827248156070709], [-0.09251242130994797, -0.30148422718048096, -0.6996451020240784, 0.605792224407196, 2.5014069080352783, 1.1586220264434814, -0.5296987295150757, -0.4218306839466095, -1.5426651239395142, -0.42894476652145386, 0.3778560161590576, -0.09585290402173996, -4.004684925079346, -0.8848157525062561, 2.620744228363037, -2.5619709491729736], [-1.143492579460144, 0.31445813179016113, 1.123146414756775, 0.5573069453239441, -0.6385922431945801, -0.3927114009857178, -0.22345533967018127, -0.2707287073135376, -2.679863214492798, 1.3003679513931274, -0.37621721625328064, 18.223346710205078, 0.2487931102514267, 0.7354050278663635, -2.3426637649536133, 0.33681902289390564], [-1.8357000350952148, -0.1330225020647049, -1.527975082397461, 1.158077359199524, 0.5179060101509094, 0.4040117859840393, -2.0589029788970947, 0.08754172921180725, 2.2630679607391357, -0.2095695436000824, -1.197335124015808, -1.321690559387207, -0.3453451693058014, 0.29363614320755005, 0.3554202914237976, 0.16128301620483398], [-0.17641329765319824, 0.4159824550151825, 1.8624681234359741, -0.876419723033905, -0.1764245182275772, -1.8324204683303833, -0.19954557716846466, 0.2799278497695923, 1.4343031644821167, -0.9653192758560181, 0.42489784955978394, 2.7669639587402344, -0.8374615907669067, 0.5849741697311401, 1.668642282485962, 1.5984328985214233], [-0.08335433900356293, 0.44659650325775146, -0.9296837449073792, -1.020898699760437, -0.40449240803718567, 0.6424883604049683, -0.5202739834785461, -0.022549279034137726, 1.2750014066696167, -0.6532115340232849, -0.7671154737472534, 1.6265320777893066, 0.8542176485061646, 0.23469595611095428, 0.04959622398018837, -1.492194414138794], [0.9522013068199158, 3.653104543685913, -1.7241082191467285, -0.34262824058532715, 0.5632832646369934, 1.5542378425598145, -0.29238566756248474, 0.06020144745707512, -0.10113899409770966, 1.2688877582550049, -0.23467673361301422, -0.3001621663570404, 7.626226425170898, -0.8988519310951233, 1.8093005418777466, 1.8122385740280151], [-0.7133926153182983, 0.04215611517429352, 0.2447877824306488, 0.5167462825775146, -2.844282627105713, -0.21939553320407867, 0.7869457602500916, 0.8839178085327148, -0.4782804250717163, -0.3596031963825226, 0.8192163705825806, 0.7057871222496033, 0.5528438091278076, -0.6523104310035706, 0.15904706716537476, -0.21525637805461884], [0.47168874740600586, -0.051881466060876846, -1.029312252998352, -2.2589492797851562, 1.8750056028366089, -3.7073400020599365, 0.40485620498657227, 0.011477276682853699, 0.16684435307979584, 0.008560780435800552, 0.37886232137680054, 7.149355411529541, 0.6578937768936157, -1.4345608949661255, -1.288514494895935, 0.3421798050403595], [-2.1381616592407227, 0.3982827961444855, 0.16462841629981995, 3.131088972091675, 7.421165466308594, -3.0073447227478027, -1.0821771621704102, 0.7981433868408203, -0.2648731768131256, -35.27217483520508, -1.375312328338623, 2.7489123344421387, -1.1282248497009277, 0.18116731941699982, -1.062819242477417, 0.7436351776123047], [0.988192617893219, 2.698843479156494, -0.018626105040311813, -0.9194472432136536, -1.307536005973816, 0.3282865285873413, -1.1477458477020264, -0.40669262409210205, 0.3047453463077545, 0.6039459705352783, 0.5845798850059509, 0.13579466938972473, 0.4736845791339874, -3.04553484916687, -1.4837671518325806, 7.935683250427246]], "dtype": "float32"}, 4], "e": {"__tup__": [{"__nd__": [[0.980944812297821, 0.980944812297821, 0.1532726287841797, -3.433306932449341, 0.5091023445129395, -2.3758108615875244, 1.3576061725616455, -0.13788187503814697, 0.6483879685401917, -4.754845142364502, 0.7024202942848206, -5.6193623542785645, 0.3990175426006317, 2.9019455909729004, 0.580389142036438, 0.020404305309057236], [1.4714171886444092, -1.3487991094589233, 1.1035629510879517, 1.1035629510879517, -0.6788030862808228, 0.7636534571647644, -0.5515275001525879, 1.018204689025879, -6.051620960235596, -1.0806466341018677, -0.9725819230079651, -0.04052424803376198, -0.36274319887161255, -2.176459312438965, -1.741167426109314, 0.7254863977432251], [0.6743995547294617, -3.433306932449341, -0.11495447158813477, 0.3678542971611023, -1.5273069143295288, 0.1909133642911911, 0.5091023445129395, -0.06363779306411743, -0.6483879685401917, -1.7290345430374146, -1.512905240058899, -0.8645172715187073, 4.643113136291504, 0.21764592826366425, 2.321556568145752, -1.886264681816101], [-0.8583267331123352, -0.08429994434118271, -1.1035629510879517, -0.24523620307445526, 0.12727558612823486, -0.5939527153968811, -0.848503828048706, 0.9333542585372925, -0.6483879685401917, 1.2967759370803833, 0.13508082926273346, 0.3512101471424103, -0.23578308522701263, 0.16323444247245789, 5.223502159118652, -8.125448226928711], [-0.6197475790977478, -1.1268137693405151, 6.310157299041748, 0.36621448397636414, -2.2636308670043945, 1.6348445415496826, -14.08481502532959, -0.15719659626483917, -0.650833785533905, 0.650833785533905, 0.4474482238292694, -1.9525012969970703, 0.2502928078174591, 0.21453669667243958, -0.5363417267799377, 0.28604891896247864], [0.22536276280879974, 2.929715871810913, 0.18310724198818207, -0.901451051235199, 0.8174222707748413, -1.2575727701187134, 1.509087324142456, -0.3458325266838074, 0.09152349829673767, -7.1591715812683105, 0.7321879863739014, 1.7897928953170776, -0.8581467866897583, -0.32180503010749817, 1.0011712312698364, -0.5005856156349182], [-0.09155362099409103, -0.3098737895488739, -0.676088273525238, 0.6197475790977478, 2.5151455402374268, 1.1318154335021973, -0.5030291080474854, -0.40871113538742065, -1.4643759727478027, -0.4474482238292694, 0.3660939931869507, -0.09152349829673767, -4.004684925079346, -0.8581467866897583, 2.5744402408599854, -2.5744402408599854], [-1.1268137693405151, 0.3098737895488739, 1.1268137693405151, 0.5634068846702576, -0.6287863850593567, -0.377271831035614, -0.22007523477077484, -0.2829538583755493, -2.60333514213562, 1.30166757106781, -0.3660939931869507, 18.223346710205078, 0.2502928078174591, 0.7151222825050354, -2.288391351699829, 0.32180503010749817], [-1.826552391052246, -0.13046802580356598, -1.5656163692474365, 1.1742122173309326, 0.5147257447242737, 0.40442734956741333, -2.0589029788970947, 0.0919153094291687, 2.1740431785583496, -0.1976402848958969, -1.185841679573059, -1.284661889076233, -0.3404565453529358, 0.30641090869903564, 0.3404565453529358, 0.15320545434951782], [-0.17939352989196777, 0.4240210950374603, 1.826552391052246, -0.8480421900749207, -0.1838306188583374, -1.7647738456726074, -0.20221367478370667, 0.2757459282875061, 1.482302188873291, -0.9882014393806458, 0.44469064474105835, 2.7669639587402344, -0.8170957565307617, 0.6128218173980713, 1.6341915130615234, 1.6341915130615234], [-0.08154251426458359, 0.4566380977630615, -0.913276195526123, -1.0437442064285278, -0.40442734956741333, 0.6617901921272278, -0.5147257447242737, -0.022978827357292175, 1.284661889076233, -0.6423309445381165, -0.7905611395835876, 1.5811222791671753, 0.885187029838562, 0.23831959068775177, 0.05106848478317261, -1.4980088472366333], [0.9785102009773254, 3.653104782104492, -1.6960843801498413, -0.35878705978393555, 0.5514918565750122, 1.6177093982696533, -0.29412898421287537, 0.059744950383901596, -0.09882014244794846, 1.284661889076233, -0.22234532237052917, -0.29646041989326477, 7.626226902008057, -0.885187029838562, 1.770374059677124, 1.770374059677124], [-0.726859986782074, 0.04193422943353653, 0.251605361700058, 0.503210723400116, -2.9154577255249023, -0.2153463214635849, 0.795124888420105, 0.8613852858543396, -0.4723952114582062, -0.35429641604423523, 0.787325382232666, 0.7085928320884705, 0.5668345093727112, -0.6376888155937195, 0.15942220389842987, -0.2125629484653473], [0.44729843735694885, -0.05241778492927551, -1.006421446800232, -2.236492156982422, 1.8552913665771484, -3.710582733154297, 0.3975624442100525, 0.01138850674033165, 0.15746507048606873, 0.00861137080937624, 0.393662691116333, 6.928462982177734, 0.6376888155937195, -1.4170862436294556, -1.275377631187439, 0.3542715609073639], [-2.236492156982422, 0.39138612151145935, 0.16773691773414612, 3.131088972091675, 7.421165466308594, -2.9154577255249023, -1.0601664781570435, 0.795124888420105, -0.255880743265152, -35.27217483520508, -1.417185664176941, 2.834371328353882, -1.1336690187454224, 0.17713578045368195, -1.062814712524414, 0.7085431218147278], [1.006421446800232, 2.683790683746338, -0.019219854846596718, -0.8945968747138977, -1.325208067893982, 0.3313020169734955, -1.1926872730255127, -0.3975624442100525, 0.29524701833724976, 0.5904940366744995, 0.5904940366744995, 0.13778193295001984, 0.46055305004119873, -3.1175897121429443, -1.4170862436294556, 7.935683250427246]], "dtype": "float32"}, {"__nd__": [[0.007663631346076727, 0.005303149111568928, 0.013508082367479801, 0.018137160688638687], [0.014085172675549984, 0.031439319252967834, 0.040677111595869064, 0.008939028717577457], [0.008154251612722874, 0.004595765378326178, 0.0061762589029967785, 0.01702282764017582], [0.006989038083702326, 0.016565101221203804, 0.07873253524303436, 0.017713578417897224]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.254629135131836, 1.1824698448181152, -1.268728494644165, 1.121245265007019, 0.06777805835008621], [-0.4914257824420929, -0.5869178771972656, -0.4497150778770447, -1.4650732278823853, 0.22763986885547638], [-1.0989888906478882, 1.1580613851547241, 0.3466879725456238, -2.203975200653076, -1.2219312191009521], [-0.1392490267753601, 0.5429985523223877, 0.18215930461883545, -0.07430271804332733, 1.471907615661621], [0.7102028727531433, 0.39612823724746704, 0.4667230248451233, 1.2311738729476929, 1.0360147953033447], [0.31074047088623047, -0.38139647245407104, 1.3493952751159668, 2.0014560222625732, -0.35597673058509827], [0.2500087022781372, 1.0978931188583374, 0.26316192746162415, 0.8769741654396057, 1.4497336149215698], [-0.9710275530815125, -0.7194429039955139, -1.5479289293289185, 0.7993379831314087, 0.07277250289916992], [-1.410115122795105, 1.8015131950378418, 0.8635219931602478, -2.1047096252441406, -0.3742450475692749]], "dtype": "float32"}, 1], "e": {"__tup__": [{"__nd__": [[1.254629135131836, 1.1824698448181152, -1.268728494644165, 1.121245265007019, 0.06777805835008621], [-0.4914257824420929, -0.5869178771972656, -0.4497150480747223, -1.4650732278823853, 0.22763986885547638], [-1.0989888906478882, 1.1580613851547241, 0.3466879725456238, -2.203975200653076, -1.2219312191009521], [-0.1392490267753601, 0.5429985523223877, 0.18215930461883545, -0.07430271804332733, 1.471907615661621], [0.7102028727531433, 0.39612823724746704, 0.4667230248451233, 1.2311738729476929, 1.0360147953033447], [0.31074047088623047, -0.38139647245407104, 1.3493952751159668, 2.0014560222625732, -0.35597673058509827], [0.2500087022781372, 1.0978931188583374, 0.26316192746162415, 0.8769741058349609, 1.4497336149215698], [-0.9710274934768677, -0.7194429039955139, -1.5479289293289185, 0.7993379831314087, 0.07277250289916992], [-1.410115122795105, 1.8015131950378418, 0.8635219931602478, -2.1047096252441406, -0.3742450475692749]], "dtype": "float32"}, {"__nd__": [[0.0028005114290863276, 0.002639441518113017, 0.002831983147189021, 0.002502779709175229, 0.00015129031089600176], [0.0010969325667247176, 0.0013100845972076058, 0.0010038282489404082, 0.0032702528405934572, 0.0005081247072666883], [0.0024531001690775156, 0.0025849584490060806, 0.0007738570566289127, 0.004919587634503841, 0.0027275250758975744], [0.00031082372879609466, 0.0012120503233745694, 0.00040660559898242354, 0.0001658542751101777, 0.0032855081371963024], [0.0015852742362767458, 0.0008842148235999048, 0.00104179244954139, 0.0027481559664011, 0.0023125330917537212], [0.0006936170975677669, 0.0008513313950970769, 0.003012042958289385, 0.004467535763978958, 0.0007945909164845943], [0.0005580551223829389, 0.0024506542831659317, 0.0005874150083400309, 0.001957531552761793, 0.0032360125333070755], [0.0021674721501767635, 0.0016058993060141802, 0.0034551986027508974, 0.0017842365195974708, 0.00016243862046394497], [0.0031475783325731754, 0.0040212348103523254, 0.00192750443238765, 0.004698012489825487, 0.0008353684097528458]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(fp8_blockwise, json.loads(_TESTS), 'torch', *[1e-06, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")